In [ ]:
!pip install langgraph pandas langchain-groq


In [ ]:

import os
from getpass import getpass
os.environ['GROQ_API_KEY'] = getpass('Enter your Groq API key:')


In [ ]:
import pandas as pd
doctors_df = pd.read_csv('doctors.csv')
# import data of doctors

from google.colab import files

uploaded = files.upload()

# Upload/select doctors.csv
doctors_df = pd.read_csv("doctors.csv")

doctors_df

In [ ]:
from typing import TypedDict, Literal

class PatientState(TypedDict):
    name: str
    age: str      # people can write age in words ex : thirty two 32
    query: str
    ward: str
    reasoning: str
    assigned_doctor: str

In [ ]:
def intake_node(state: PatientState) -> PatientState:
    print("=== Patient Intake Form ===")

    name = input("Patient name: ")
    age = input("Patient age: ")
    query = input("Describe the problem/symptoms: ")

    return {
        "name": name,
        "age": age,
        "query": query,
        "ward": "",
        "reasoning": "",
        "assigned_doctor": "",
    }


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

ROUTER_SYSTEM_PROMPT = """You are a hospital triage router. Based on the patient's
description, classify them into EXACTLY ONE of these wards:

- emergency: life-threatening or urgent physical conditions (severe pain, bleeding,
  breathing difficulty, chest pain, accidents, high fever, unconsciousness, etc.)
- mental_health: psychological/emotional distress (anxiety, depression, panic attacks,
  suicidal thoughts, self-harm, trauma, etc.)
- general: everything else (routine checkups, mild/chronic non-urgent symptoms, colds,
  minor aches, follow-ups, etc.)

Respond with ONLY one word: emergency, mental_health, or general.
"""

# Hard safety-net keywords — these always override the LLM, since misrouting
# a genuine crisis is unacceptable. Good example of "guardrail before AI".
EMERGENCY_KEYWORDS = ["chest pain", "can't breathe", "unconscious", "severe bleeding", "heart attack"]
CRISIS_KEYWORDS = ["suicide", "kill myself", "self-harm", "want to die"]

def router_node(state: PatientState) -> PatientState:
    query_lower = state["query"].lower()

    if any(kw in query_lower for kw in CRISIS_KEYWORDS):
        ward, reasoning = "mental_health", "Crisis keyword detected — routed immediately for safety."
    elif any(kw in query_lower for kw in EMERGENCY_KEYWORDS):
        ward, reasoning = "emergency", "Emergency keyword detected — routed immediately for safety."
    else:
        response = llm.invoke([
            SystemMessage(content=ROUTER_SYSTEM_PROMPT),
            HumanMessage(content=f"Age: {state['age']}\nSymptoms: {state['query']}")
        ])
        raw = response.content.strip().lower()

        # Defensive parsing in case the model adds extra words
        if "emergency" in raw:
            ward = "emergency"
        elif "mental_health" in raw or "mental health" in raw:
            ward = "mental_health"
        elif "general" in raw:
            ward = "general"
        else:
            ward = "general"  # safe default

        reasoning = f"LLM classified based on: '{state['query']}'"

    return {**state, "ward": ward, "reasoning": reasoning}

In [ ]:
def general_ward_node(state: PatientState) -> PatientState:
    print(f"\n{state['name']} -> GENERAL WARD")
    print(f"Reason: {state['reasoning']}")
    return state


def emergency_ward_node(state: PatientState) -> PatientState:
    print(f"\n{state['name']} -> EMERGENCY WARD (priority)")
    print(f"Reason: {state['reasoning']}")
    return state


def mental_health_ward_node(state: PatientState) -> PatientState:
    print(f"\n{state['name']} -> MENTAL HEALTH WARD")
    print(f"Reason: {state['reasoning']}")
    return state

In [ ]:
def doctor_availability_node(state: PatientState) -> PatientState:
    ward_from_state = state["ward"].lower().strip()
    print(f"DEBUG: Processing patient for ward: '{ward_from_state}'")

    # Ensure columns are string type before applying string methods
    doctors_df['ward'] = doctors_df['ward'].astype(str)
    doctors_df['status'] = doctors_df['status'].astype(str)

    # Filter for matching ward and active status
    matching_ward_doctors = doctors_df[doctors_df["ward"].str.lower().str.strip() == ward_from_state]
    available = matching_ward_doctors[matching_ward_doctors["status"].str.lower().str.strip() == "active"]
    print(f"DEBUG: Available doctors for ward '{ward_from_state}' with status 'active':\n{available.to_string()}")

    if not available.empty:
        doctor = available.iloc[0]["doctor_name"]
        assigned_doctor = doctor
        # Mark doctor busy for the rest of this Colab session
        doctors_df.loc[doctors_df["doctor_name"] == doctor, "status"] = "busy"
        print(f"DEBUG: Doctors_df status updated for '{doctor}':\n{doctors_df[doctors_df['doctor_name'] == doctor].to_string()}")
    else:
        assigned_doctor = "No doctor currently free" # Strict adherence to user's requested message

    print(f"👨‍⚕️ Doctor assigned: {assigned_doctor}")
    return {**state, "assigned_doctor": assigned_doctor}

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
import pandas as pd # Added import for pandas

# Re-defining PatientState for self-containment of this cell
class PatientState(TypedDict):
    name: str
    age: str
    query: str
    ward: str
    reasoning: str
    assigned_doctor: str

# Re-defining doctor_availability_node for self-containment of this cell
def doctor_availability_node(state: PatientState) -> PatientState:
    ward_from_state = state["ward"].lower().strip()
    print(f"DEBUG: Processing patient for ward: '{ward_from_state}'")

    # Ensure columns are string type before applying string methods
    doctors_df['ward'] = doctors_df['ward'].astype(str)
    doctors_df['status'] = doctors_df['status'].astype(str)

    # Filter for matching ward and active status
    matching_ward_doctors = doctors_df[doctors_df["ward"].str.lower().str.strip() == ward_from_state]
    available = matching_ward_doctors[matching_ward_doctors["status"].str.lower().str.strip() == "active"]
    print(f"DEBUG: Available doctors for ward '{ward_from_state}' with status 'active':\n{available.to_string()}")

    if not available.empty:
        doctor = available.iloc[0]["doctor_name"]
        assigned_doctor = doctor
        # Mark doctor busy for the rest of this Colab session
        doctors_df.loc[doctors_df["doctor_name"] == doctor, "status"] = "busy"
        print(f"DEBUG: Doctors_df status updated for '{doctor}':\n{doctors_df[doctors_df['doctor_name'] == doctor].to_string()}")
    else:
        assigned_doctor = "No doctor currently free" # Strict adherence to user's requested message

    print(f"👨‍⚕️ Doctor assigned: {assigned_doctor}") # Completed the print statement
    return {**state, "assigned_doctor": assigned_doctor}

def route_decision(state: PatientState) -> Literal["general", "emergency", "mental_health"]:
    return state["ward"]

builder = StateGraph(PatientState)

builder.add_node("intake", intake_node)
builder.add_node("router", router_node)
builder.add_node("general", general_ward_node)
builder.add_node("emergency", emergency_ward_node)
builder.add_node("mental_health", mental_health_ward_node)
builder.add_node("doctor_check", doctor_availability_node)

builder.set_entry_point("intake")
builder.add_edge("intake", "router")

builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "general": "general",
        "emergency": "emergency",
        "mental_health": "mental_health",
    },
)

builder.add_edge("general", "doctor_check")
builder.add_edge("emergency", "doctor_check")
builder.add_edge("mental_health", "doctor_check")

builder.add_edge("doctor_check", END)

graph = builder.compile()

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Example: Running the graph
print("Starting the hospital triage process...")
result = graph.invoke({})
print("\n--- Triage Result ---")
print(f"Patient: {result['name']}")
print(f"Age: {result['age']}")
print(f"Symptoms: {result['query']}")
print(f"Routed to Ward: {result['ward']} ({result['reasoning']})")
print(f"Assigned Doctor: {result['assigned_doctor']}")

In [ ]:
final_state = graph.invoke({})
print("\n=== Final Summary ===")
print(f"Patient: {final_state['name']} (Age: {final_state['age']})")
print(f"Ward: {final_state['ward']}")
print(f"Assigned Doctor: {final_state['assigned_doctor']}")

In [ ]:
doctors_df
